In [ ]:
!pip install selenium webdriver-manager

In [ ]:
!pip install pandas
!pip install beautifulsoup4
!pip install selenium webdriver-manager
!pip install selenium

In [2]:
!pip install aiohttp


In [1]:
import asyncio
import aiohttp
import json
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys
import nest_asyncio
nest_asyncio.apply()

# --- Funções Auxiliares de Limpeza ---

def limpar_preco(texto):
    if not texto: 
        return "0"
    # Remove 'R$', espaços em branco e caracteres invisíveis de formatação (\xa0)
    return texto.replace("R$", "").replace("\xa0", "").replace(" ", "").strip()

def limpar_porcentagem(texto):
    if not texto: 
        return "0"
    # Remove '%', sinal negativo (se houver) e espaços
    return texto.replace("%", "").replace("-", "").replace(" ", "").strip()


# --- Funçoes de Extração ---

def achar_EAN_gtin(soup):
    try:
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            if script.string:
                dados_json = json.loads(script.string)
                itens = dados_json if isinstance(dados_json, list) else [dados_json]
                for item in itens:
                    if item.get('@type') == 'Product' and 'gtin13' in item:
                        return item['gtin13']
        return "NA"
    except:
        return "NA"

def achar_nome(soup):
    try: return soup.select_one("meta[property='og:title']")["content"].strip()
    except: return "NA"

def achar_marca(soup):
    try: return soup.select_one(".title_marca").get_text(strip=True)
    except: return "NA"

def achar_preco_antes_desconto(soup):
    try: 
        texto = soup.select_one(".unit-price").get_text(strip=True)
        return limpar_preco(texto)
    except: return "0"

def achar_desconto(soup):
    try: 
        texto = soup.select_one(".discount").get_text(strip=True)
        return limpar_porcentagem(texto)
    except: return "NA"

def achar_preco(soup):
    try: 
        texto = soup.select_one("p.sale-price:not(.sale-price-pix)").get_text(strip=True)
        return limpar_preco(texto)
    except: return "0"

def achar_pix(soup):
    try: 
        texto = soup.select_one(".sale-price-pix").get_text(strip=True)
        return limpar_preco(texto)
    except: return "0"

def achar_cashback(soup):
    try: 
        texto = soup.select_one("strong.loyalty_price").get_text(strip=True)
        return limpar_preco(texto)
    except: return "NA"

def achar_apenas_pix(soup):
    try:
        selo_pix = soup.select_one(".seal-desconto-pix")
        if selo_pix and "DESCONTO NO PIX" in selo_pix.get_text(strip=True).upper():
            return "Sim"
        return "NA"
    except:
        return "NA"

def achar_promo_volume(soup):
    try:
        for b in soup.find_all('b'):
            texto = b.get_text(strip=True)
            if "A PARTIR DE" in texto.upper() and "UNIDADES" in texto.upper():
                return " ".join(texto.split()) 
        return "NA"
    except:
        return "NA"

# --- Async: Processamento de Produtos ---

async def processar_produto(session, url, semaphore):
    async with semaphore:
        try:
            async with session.get(url, timeout=15) as response:
                html = await response.text()
                soup = BeautifulSoup(html, "html.parser")

                nome = achar_nome(soup)
                ean = achar_EAN_gtin(soup)
                
                # Organização
                item = {
                    "Categoria": "Medicamentos",
                    "Nome": nome,
                    "Marca": achar_marca(soup),
                    "EAN": ean,
                    "Preço sem desconto": achar_preco_antes_desconto(soup),
                    "Desconto (%)": achar_desconto(soup),
                    "Preço com Desconto": achar_preco(soup),
                    "Preço Pix": achar_pix(soup),
                    "Cashback": achar_cashback(soup),
                    "Apenas PIX": achar_apenas_pix(soup),
                    "Promoção por Volume": achar_promo_volume(soup),
                    "Link": url
                }

                #só para confirmar
                print(f" [OK] {nome[:35]:<35} | EAN: {ean:<13}")
                
                return item
        except Exception as e:
            print(f" [ERRO] Link: {url} | Motivo: {e}")
            return None

# --- async loop ---

async def main():
    url_base = "https://www.farmaponte.com.br/saude/medicamentos/?p={}"
    pagina = 1
    links_produtos = []
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    async with aiohttp.ClientSession(headers=headers) as session:
        # 1. Coleta de dados
        print("MAPEANDO AS PÁGINAS DO SITE")
        while True:
            try:
                async with session.get(url_base.format(pagina)) as response:
                    if response.status != 200:
                        break
                    html = await response.text()
                
                soup = BeautifulSoup(html, "html.parser")
                produtos = soup.select(".item-product")
                
                if not produtos: 
                    print(f"\nFim da paginação na página {pagina-1}.")
                    break
                
                for p in produtos:
                    try: 
                        href = p.select_one(".title a")['href']
                        link_completo = f"https://www.farmaponte.com.br{href}" if href.startswith("/") else href
                        links_produtos.append(link_completo)
                    except: pass
                
                print(f"Lendo página {pagina}... Total acumulado: {len(links_produtos)} links", end="\r")
                pagina += 1
            except:
                break

        print(f"\n\nTotal de produtos para processar: {len(links_produtos)}")
        print("-" * 70)

        # 2. Async
        semaphore = asyncio.Semaphore(15) 
        tarefas = [processar_produto(session, link, semaphore) for link in links_produtos]
        
        resultados = await asyncio.gather(*tarefas)
        
        # Filtro
        dados_finais = [r for r in resultados if r is not None]

    # --- TABELA E TRATAMENTO FINAL ---
    if dados_finais:
        df = pd.DataFrame(dados_finais)
        colunas_ordem = [
            "Categoria", "Nome", "Marca", "EAN", 
            "Preço sem desconto", "Desconto (%)", "Preço com Desconto", 
            "Preço Pix", "Cashback", "Apenas PIX", "Promoção por Volume", "Link"
        ]
        df = df[colunas_ordem]
        
        # Remover produtos duplicados com base no Link
        qnt_antes = len(df)
        df.drop_duplicates(subset=["Link"], keep="first", inplace=True)
        qnt_depois = len(df)
        duplicados_removidos = qnt_antes - qnt_depois
        
        df.to_excel("medicamentos_completo.xlsx", index=False)
        
        print("\n" + "="*50)
        print(f"Foram raspados {qnt_antes} produtos no total.")
        if duplicados_removidos > 0:
            print(f"Foram removidos {duplicados_removidos} produtos duplicados.")
        print(f"Total salvo no Excel: {qnt_depois} produtos únicos.")
        print("Arquivo: 'medicamentos_completo.xlsx' gerado com sucesso.")
        print("="*50)
    else:
        print("Nenhum dado foi coletado.")

if __name__ == "__main__":
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    
    start_time = time.time()
    asyncio.run(main())
    print(f"Tempo total de execução: {time.time() - start_time:.2f}s")

MAPEANDO AS PÁGINAS DO SITE
Lendo página 267... Total acumulado: 5332 links

Total de produtos para processar: 5332
----------------------------------------------------------------------
 [OK] Citrato De Sildenafila Eurofarma 50 | EAN: 7891317477592
 [OK] Dulcolax 5Mg, Caixa Com 20 Drageas  | EAN: 7891058021528
 [OK] Alenia em Pó de Uso Inalatório 12mc | EAN: 7896181928591
 [OK] Odorizante De Ambiente Bendita Canf | EAN: 7896186002548
 [OK] Citrato De Sildenafila Medley 50Mg, | EAN: 7896422525190
 [OK] Tadalafila Eurofarma 20Mg, Caixa Co | EAN: 7891317127800
 [OK] Colágeno Condres 90 Cápsulas        | EAN: 7894916515207
 [OK] Simbioflora 15 Saches Com 6G De Po  | EAN: 7898040323806
 [OK] Tadalafila Eurofarma 5Mg, Caixa Com | EAN: 7891317127725
 [OK] Dipirona Monoidratada Solução de Us | EAN: 7896181911036
 [OK] Sedatif Pc Com 60 Comprimidos       | EAN: 7898927111014
 [OK] Dipirona Gotas 10ml Germed          | EAN: 7896004719115
 [OK] Permanganato De Potassio 10 Comprim | EAN: 78911370

In [2]:
!pip install nest_asyncio